# MDSP Project - Matrix Vector Multiplication


## 1. Load Overlay

In [ ]:
#import global_state as gs
#gs.clear_global_state()
print("Done")
import time
import struct
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("matmulcheck.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
matmul_stream  = ol.matmul_stream_2_0
fir_mmio_ip = ol.fir_mmio_0
hw_timer    = ol.axi_timer_0

## 2. Prepare Test Signal



In [ ]:
import sys

import os
import numpy as np

# -----------------------------
# Normalization helpers
# -----------------------------
def normalize_vector(x):
    norm = np.linalg.norm(x)
    return x if norm == 0 else x / norm

def normalize_matrix(A):
    norm = np.linalg.norm(A)
    return A if norm == 0 else A / norm

# -----------------------------
# Generate vectors
# -----------------------------
def generate_vectors(n, ns, save = True):
    os.makedirs("test/input", exist_ok=True)

    X = np.random.randn(ns, n)
    X = np.array([normalize_vector(sample) for sample in X])

    if save:
        path = f"test/input/{n}.txt"
        np.savetxt(path, X, fmt="%.6f")

        print(f"{ns} input sample(s) saved to {path}")
    return X

# -----------------------------
# Generate matrix
# -----------------------------
def generate_matrix(n, save = True):
    os.makedirs("test/matrices", exist_ok=True)

    A = np.random.randn(n, n)
    A = normalize_matrix(A)

    if save:
        path = f"test/matrices/{n}.txt"
        np.savetxt(path, A, fmt="%.6f")

        print(f"Matrix saved to {path}")
    return A

# -----------------------------
# Main pipeline
# -----------------------------
def generate_all(n, ns, save = True):
    # Generate inputs
    A = generate_matrix(n, save)
    X = generate_vectors(n, ns, save)

    # Compute Y = A * X for each sample
    Y = X @ A.T

    if save:
        os.makedirs("test/output", exist_ok=True)
        out_path = f"test/output/{n}.txt"
        np.savetxt(out_path, Y, fmt="%.6f")

        print(f"Output (Y = AX) saved to {out_path}")
    return A, X, Y

def format_input_data(x, FRACTIONAL_BITS=20, USE_FIXED=False):
    """
    Convert float array x to fixed-point *raw codes* suitable for streaming via DMA.

    - If USE_FIXED=False: returns float32 array (unchanged).
    - If USE_FIXED=True : returns int32 array where each element is the raw fixed-point
      code q = round(x * 2^F), saturated to TOTAL_BITS signed range.
      These codes can be written into a 32-bit AXI-stream word (LSBs hold the fixed code).
    """
    TOTAL_BITS=32 
    signed=True
    x = np.asarray(x)

    if not USE_FIXED:
        return x.astype(np.float32, copy=False)

    F = int(FRACTIONAL_BITS)
    scale = 1 << F

    # quantize
    q = np.rint(x.astype(np.float64) * scale).astype(np.int64)

    # saturation limits for TOTAL_BITS fixed storage
    if signed:
        qmin = -(1 << (TOTAL_BITS - 1))
        qmax =  (1 << (TOTAL_BITS - 1)) - 1
    else:
        qmin = 0
        qmax = (1 << TOTAL_BITS) - 1

    q = np.clip(q, qmin, qmax)

    # Return as int32 since DMA buffers commonly use 32-bit lanes
    return q.astype(np.int32)

def format_output_data(x, FRACTIONAL_BITS=20, USE_FIXED=False):
    """
    Convert DMA-received output buffer to float values.

    - If USE_FIXED=False: interpret x as float32 samples (already correct).
    - If USE_FIXED=True : interpret x as int32 fixed-point *raw codes* (LSBs contain code),
      and convert back to float: y = q / 2^F.
    """
    x = np.asarray(x)

    if not USE_FIXED:
        return x.astype(np.float32, copy=False)

    F = int(FRACTIONAL_BITS)
    scale = float(1 << F)

    # DMA buffer likely int32, containing signed codes (sign-extended)
    q = x.astype(np.int32, copy=False)

    # Convert code -> float
    return (q.astype(np.float32) / scale)


In [48]:
import numpy as np

N = 64                  # Matrix size
NS = 100                  # Number of x samples
USE_FIXED = True
FRACTIONAL_BITS = 26

matrix_data, input_vectors, output_vectors = generate_all(N, NS, False)

matrix_data = np.array(matrix_data, dtype=np.float32)
input_vectors = np.array(input_vectors, dtype=np.float32).reshape(NS, N)
output_vectors = np.array(output_vectors, dtype=np.float32).reshape(NS, N)

print(matrix_data.shape)
print(input_vectors.shape)
print(output_vectors.shape)

N_INPUT_WORDS = N * N + NS * N   # matrix + all input vectors
N_OUTPUT_WORDS = NS * N

matrix = matrix_data.reshape(N, N)
matrix_flat = matrix.flatten()
inputs_flat = input_vectors.reshape(NS * N)

full_input = np.concatenate([matrix_flat, inputs_flat]).astype(np.float32)

# convert input data for FIXED point
full_input = format_input_data(full_input, FRACTIONAL_BITS, USE_FIXED)


## 4. DMA Streaming FIR

Data path: PS $\rightarrow$ DMA TX $\rightarrow$ AXI-Stream $\rightarrow$ FIR $\rightarrow$ AXI-Stream $\rightarrow$ DMA RX $\rightarrow$ PS

The CPU only sets up the transfer and waits -- the FPGA processes all samples autonomously.

In [ ]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles
in_buf.freebuffer()
# out_buf.freebuffer()

In [ ]:
dma_dtype = np.int32 if USE_FIXED else np.float32
in_buf  = allocate(shape=(N_INPUT_WORDS,), dtype=dma_dtype)
out_buf = allocate(shape=(N_OUTPUT_WORDS,), dtype=dma_dtype)

np.copyto(in_buf, full_input)
out_buf[:] = 0

# Configure streaming matmul: set sample count and start
# Register 0x10 = 'n' parameter (check synthesis report)
matmul_stream.write(0x10, NS)
matmul_stream.write(0x00, 0x01)   # ap_start

t0 = time.perf_counter()
timer_start(hw_timer)

dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()

cycles = timer_stop(hw_timer)
t_dma = time.perf_counter() - t0

y_dma = np.array(out_buf, dtype=dma_dtype)

# convert received data for FIXED point
y_dma = format_output_data(y_dma, FRACTIONAL_BITS, USE_FIXED)

print(f"DMA: {NS} sample(s), {N_INPUT_WORDS} input words, {N_OUTPUT_WORDS} output words in {t_dma*1e3:.2f} ms "
      f"({t_dma/NS*1e6:.1f} us/sample)")
print(f"HW timer: {cycles} cycles = {cycles/FCLK_MHZ:.1f} us @ {FCLK_MHZ:.0f} MHz")
print(f"Latency per sample = {(cycles/FCLK_MHZ)/NS:.1f}")

In [51]:
y_calculated = np.reshape(y_dma, (NS, N))
error = y_calculated - output_vectors
l2_errors = np.linalg.norm(error, axis=1)
avg_l2_error = np.mean(l2_errors)

# print(y_calculated)
print("Average L2 error:", avg_l2_error)



Average L2 error: 3.8227627e-06


In [ ]:
# TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
# FCLK_MHZ = 100.0

# def timer_start(tmr):
#     tmr.write(TLR0, 0)
#     tmr.write(TCSR0, 0x020)   # load
#     tmr.write(TCSR0, 0x080)   # enable, count up

# def timer_stop(tmr):
#     cycles = tmr.read(TCR0)
#     tmr.write(TCSR0, 0x000)
#     return cycles
# in_buf.freebuffer()
# out_buf.freebuffer()

In [ ]:
# Re-run DMA test with HW timer
np.copyto(in_buf, full_input)
out_buf[:] = 0

matmul_stream.write(0x10, 0x01)
matmul_stream.write(0x00, 0x01)

timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

print(f"HW timer: {cycles} cycles = {cycles/FCLK_MHZ:.1f} us @ {FCLK_MHZ:.0f} MHz")
print(f"Latency per sample = {(cycles/FCLK_MHZ)/NS:.1f}")